# Faruq-v3 — FC-STB frequency-consistent distillation
Fail-fast: static audit → frozen AF2 teacher-headroom diagnostic → matched FCT0/FCD1 training. Validation only; test is never extracted.

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)
import os, shutil, subprocess, sys, time, json
from pathlib import Path
REPO=Path('/content/coffee-bean-detection'); BRANCH='agent/fc-stb-frequency-distillation'
if (REPO/'.git').is_dir():
 subprocess.run(['git','fetch','origin',BRANCH],cwd=REPO,check=True); subprocess.run(['git','checkout',BRANCH],cwd=REPO,check=True); subprocess.run(['git','reset','--hard',f'origin/{BRANCH}'],cwd=REPO,check=True)
else:
 if REPO.exists(): shutil.rmtree(REPO)
 command=['git','clone','--depth','1','--branch',BRANCH,'https://github.com/ediprin/coffee-bean-detection.git',str(REPO)]
 for attempt in range(1,4):
  result=subprocess.run(command)
  if result.returncode==0: break
  if REPO.exists(): shutil.rmtree(REPO)
  if attempt==3: raise RuntimeError('Git clone gagal tiga kali')
  time.sleep(2)
subprocess.run([sys.executable,'-m','pip','install','-q','-e',str(REPO)],check=True)
for key in list(sys.modules):
 if key=='coffee_detector' or key.startswith('coffee_detector.'): sys.modules.pop(key,None)
sys.path.insert(0,str(REPO/'src'))
print('COMMIT:',subprocess.check_output(['git','rev-parse','HEAD'],cwd=REPO,text=True).strip())

In [ ]:
import tarfile, torch
from coffee_detector.drive_project import require_project_artifact, resolve_drive_project_root
assert torch.cuda.is_available(),'Aktifkan T4 GPU.'
REQUIRED=(
 'bundles/faruq-development-v3-grouped.tar',
 'experiments/faruq-v3-breadth-screening-batch-v1/candidates/STB1/STB1_seed42/weights/best.pt',
 'experiments/faruq-v3-breadth-screening-batch-v1/candidates/STB1/val_reports/stb_seed42_screening.json',
 'experiments/faruq-v3-breadth-screening-batch-v1/candidates/AFAB/AF2_seed42/weights/best.pt',)
PROJECT_ROOT=resolve_drive_project_root(required_relative_paths=REQUIRED)
ARCHIVE=require_project_artifact(PROJECT_ROOT,REQUIRED[0]); STB_CHECKPOINT=require_project_artifact(PROJECT_ROOT,REQUIRED[1]); STB_SUMMARY=require_project_artifact(PROJECT_ROOT,REQUIRED[2]); AF2_CHECKPOINT=require_project_artifact(PROJECT_ROOT,REQUIRED[3])
DATA_ROOT=Path('/content/faruq-development-v3-grouped')
if not (DATA_ROOT/'data.yaml').is_file():
 with tarfile.open(ARCHIVE,'r') as archive: archive.extractall('/content',filter='data')
GROUPED_SUMMARY=DATA_ROOT/'faruq_grouped_summary.json'; assert GROUPED_SUMMARY.is_file(); assert not (DATA_ROOT/'test').exists()
OUTPUT_ROOT=PROJECT_ROOT/'experiments/faruq-v3-fcstb-distillation-v1'; OUTPUT_ROOT.mkdir(parents=True,exist_ok=True)
COMMON=['--data-root',str(DATA_ROOT),'--grouped-summary',str(GROUPED_SUMMARY),'--stb-summary',str(STB_SUMMARY),'--stb-checkpoint',str(STB_CHECKPOINT),'--af2-checkpoint',str(AF2_CHECKPOINT),'--output-root',str(OUTPUT_ROOT),'--seed','42','--device','0']
print('GPU:',torch.cuda.get_device_name(0)); print('PROJECT:',PROJECT_ROOT); print('OUTPUT:',OUTPUT_ROOT)

## 1. Static gate — tanpa dataset/training

In [ ]:
command=[sys.executable,'-u','-m','coffee_detector.experiments.run_faruq_v3_fcstb',*COMMON,'--stage','static']
subprocess.run(command,cwd=REPO,check=True)
static=json.loads((OUTPUT_ROOT/'static_audit.json').read_text()); print('STATIC:',static['decision']); print(static['arms']); assert static['decision']=='PASS'

## 2. Frozen teacher-headroom diagnostic — sekitar satu evaluasi, tanpa training

In [ ]:
command=[sys.executable,'-u','-m','coffee_detector.experiments.run_faruq_v3_fcstb',*COMMON,'--stage','diagnostic']
subprocess.run(command,cwd=REPO,check=True)
DIAG=OUTPUT_ROOT/'val_reports/frequency_teacher_headroom_seed42.json'; diagnostic=json.loads(DIAG.read_text()); print(json.dumps({k:diagnostic[k] for k in ('counts','rates','af2_rescue_by_class','criteria','decision')},indent=2)); assert diagnostic['decision']=='PASS','STOP: AF2 tidak memiliki rescue signal; jangan training.'

## 3. Matched training — FCT0 vs FCD1, maksimal 20 epoch masing-masing dan resume dari Drive

In [ ]:
command=[sys.executable,'-u','-m','coffee_detector.experiments.run_faruq_v3_fcstb',*COMMON,'--stage','train','--authorize-training']
print('MENJALANKAN:',' '.join(command),flush=True); subprocess.run(command,cwd=REPO,check=True)

In [ ]:
import pandas as pd
from IPython.display import display
SUMMARY=OUTPUT_ROOT/'val_reports/fcstb_seed42_decision.json'; result=json.loads(SUMMARY.read_text())
rows=[{'model':'STB1',**result['reference']['STB1']},*[{'model':k,**v} for k,v in result['candidates'].items()]]
display(pd.DataFrame(rows).style.format({c:'{:.2%}' for c in ('macro_map50_95','bottom3_class_map50_95','worst_class_map50_95')}))
print('COMPARISONS:',json.dumps(result['comparisons'],indent=2)); print('DECISION:',result['decision']); print('NEXT:',result['next_action']); print('TEST OPENED:',result['test_opened']); print('Kirim tabel dan keputusan. Jangan membuka test atau seed tambahan.')